# Control Comparison

**Forensic question:** Is the target author's convergence signal anomalous relative to a peer control author who has similar publishing volume?

**Inputs:**
- `data/analysis/comparison_report.json` (loaded if present, generated otherwise)
- `data/analysis/{slug}_convergence.json` for both target and control
- `data/analysis/{slug}_drift.json` for both target and control

**Outputs (in-notebook):**
- Side-by-side change-point timelines (target vs control)
- Per-feature-family contrast bar chart (significant ones)
- Velocity-trajectory overlay (centroid drift over time)
- Verdict: count of feature families where target activity is significantly higher than control

**Run metadata:** (auto-populated by first code cell)


In [ ]:
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype+notebook_connected+png'


In [ ]:
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype+notebook_connected+png'


In [ ]:
# Parameters — override via: quarto render NOTEBOOK -P target_slug:foo -P control_slug:bar
author_slug = "all"
target_slug = "colby-hall"
control_slug = "sarah-rumpf"


In [ ]:
# | echo: false
import sys
from datetime import UTC, datetime

from IPython.display import Markdown, display

from forensics.config import get_settings
from forensics.utils.provenance import compute_config_hash, compute_corpus_hash

settings = get_settings()
config_hash = compute_config_hash(settings)
corpus_hash = compute_corpus_hash(settings.db_path)
run_timestamp = datetime.now(UTC).isoformat()
display(
    Markdown(f"""
| Key | Value |
|-----|-------|
| Config hash | `{config_hash}` |
| Corpus hash | `{corpus_hash}` |
| Run timestamp | `{run_timestamp}` |
| Target | `{target_slug}` |
| Control | `{control_slug}` |
| Python | `{sys.version}` |
""")
)


In [ ]:
"""Load `comparison_report.json` if it covers the target/control pair, otherwise generate it."""

import json

from IPython.display import Markdown, display

from forensics.analysis.comparison import compare_target_to_controls
from forensics.config import get_project_root
from forensics.paths import AnalysisArtifactPaths

root = get_project_root()
paths = AnalysisArtifactPaths(
    project_root=root,
    db_path=settings.db_path,
    features_dir=root / "data" / "features",
    embeddings_dir=root / "data" / "embeddings",
    analysis_dir=root / "data" / "analysis",
)

report_path = paths.comparison_report_json()


def _has_pair(payload: dict, target: str, control: str) -> bool:
    targets = payload.get("targets", {}) if isinstance(payload, dict) else {}
    block = targets.get(target)
    if not isinstance(block, dict):
        return False
    return control in (block.get("control_change_points") or {})


payload: dict
generated_now = False
if report_path.is_file():
    try:
        payload = json.loads(report_path.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        payload = {"targets": {}}
else:
    payload = {"targets": {}}

if not _has_pair(payload, target_slug, control_slug):
    print(
        f"comparison_report.json does not contain pair ({target_slug} vs {control_slug}); "
        "generating in-notebook (typically <30s)..."
    )
    fresh = compare_target_to_controls(
        target_id=target_slug,
        control_ids=[control_slug],
        paths=paths,
        settings=settings,
    )
    payload.setdefault("targets", {})[target_slug] = fresh
    report_path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    generated_now = True

target_block = payload["targets"][target_slug]
feature_comparisons = target_block.get("feature_comparisons", {}) or {}
control_change_points_by_slug = target_block.get("control_change_points", {}) or {}
control_drift_scores_by_slug = target_block.get("control_drift_scores", {}) or {}
editorial_vs_author_signal = float(target_block.get("editorial_vs_author_signal") or 0.0)

display(
    Markdown(
        f"""
**Report path:** `{report_path}`
**Source:** {"generated this run" if generated_now else "loaded from disk"}
**Available controls in report:** `{list(control_change_points_by_slug.keys())}`
**Editorial-vs-author signal:** `{editorial_vs_author_signal:.3f}` _(0 = outlet-wide; 1 = author-specific)_
**Compared features (n):** `{len(feature_comparisons)}`
"""
    )
)


In [ ]:
"""Side-by-side change-point timelines: target vs control.

Each subplot shows monthly change-point counts (per the cached `_changepoints.json`
artifacts). Vertical separation makes it clear whether the target's burst pattern
is mirrored by the control or is genuinely anomalous.
"""

import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots

from forensics.utils.charts import register_forensics_template

register_forensics_template()


def _monthly_cp_counts(slug: str) -> pl.DataFrame:
    p = paths.changepoints_json(slug)
    if not p.is_file():
        return pl.DataFrame({"month": [], "n": []})
    rows = json.loads(p.read_text(encoding="utf-8"))
    if not rows:
        return pl.DataFrame({"month": [], "n": []})
    return (
        pl.DataFrame(rows)
        .with_columns(
            pl.col("timestamp")
            .str.to_datetime(format="%Y-%m-%dT%H:%M:%S%#z", strict=False)
            .dt.replace_time_zone(None)
            .dt.truncate("1mo")
            .alias("month")
        )
        .group_by("month")
        .agg(pl.len().alias("n"))
        .sort("month")
    )


target_cp = _monthly_cp_counts(target_slug)
control_cp = _monthly_cp_counts(control_slug)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        f"Target: {target_slug} — change-points per month",
        f"Control: {control_slug} — change-points per month",
    ),
    vertical_spacing=0.12,
)
if not target_cp.is_empty():
    fig.add_trace(
        go.Bar(
            x=target_cp["month"].to_list(),
            y=target_cp["n"].to_list(),
            name=target_slug,
            marker_color="#d62728",
        ),
        row=1,
        col=1,
    )
if not control_cp.is_empty():
    fig.add_trace(
        go.Bar(
            x=control_cp["month"].to_list(),
            y=control_cp["n"].to_list(),
            name=control_slug,
            marker_color="#1f77b4",
        ),
        row=2,
        col=1,
    )
fig.update_layout(
    title="Change-point density timeline (target vs control)",
    height=600,
    showlegend=False,
    bargap=0.05,
)
fig.update_xaxes(title_text="Month", row=2, col=1)
fig.update_yaxes(title_text="# change-points", row=1, col=1)
fig.update_yaxes(title_text="# change-points", row=2, col=1)
fig.show()


In [ ]:
"""Per-feature-family contrast: target sig-count vs control sig-count.

For each of the six feature families (`ai_markers`, `entropy`,
`lexical_richness`, `readability`, `self_similarity`, `sentence_structure`),
count significant features per side using each author's
`_hypothesis_tests.json` (corrected_p_value < 0.05 AND `significant` == True).
"""

from forensics.analysis.feature_families import FEATURE_FAMILIES

FAMILIES_ORDERED = [
    "ai_markers",
    "entropy",
    "lexical_richness",
    "readability",
    "self_similarity",
    "sentence_structure",
]


def _family_sig_counts(slug: str) -> dict[str, int]:
    p = paths.hypothesis_tests_json(slug)
    counts = dict.fromkeys(FAMILIES_ORDERED, 0)
    if not p.is_file():
        return counts
    rows = json.loads(p.read_text(encoding="utf-8"))
    if not rows:
        return counts
    df = pl.DataFrame(rows)
    if df.is_empty() or "feature_name" not in df.columns:
        return counts
    sig = df.filter(
        (pl.col("corrected_p_value") < 0.05) & (pl.col("significant") == True)  # noqa: E712
    )
    if sig.is_empty():
        return counts
    family_col = sig["feature_name"].map_elements(
        lambda f: FEATURE_FAMILIES.get(f, "unknown"), return_dtype=pl.Utf8
    )
    sig = sig.with_columns(family_col.alias("family"))
    grouped = sig.group_by("family").agg(pl.len().alias("n"))
    for row in grouped.iter_rows(named=True):
        if row["family"] in counts:
            counts[row["family"]] = int(row["n"])
    return counts


target_counts = _family_sig_counts(target_slug)
control_counts = _family_sig_counts(control_slug)

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=FAMILIES_ORDERED,
        y=[target_counts[f] for f in FAMILIES_ORDERED],
        name=f"Target ({target_slug})",
        marker_color="#d62728",
    )
)
fig.add_trace(
    go.Bar(
        x=FAMILIES_ORDERED,
        y=[control_counts[f] for f in FAMILIES_ORDERED],
        name=f"Control ({control_slug})",
        marker_color="#1f77b4",
    )
)
fig.update_layout(
    title="Significant features per feature family (corrected p < 0.05)",
    barmode="group",
    yaxis_title="# significant features",
    xaxis_title="Feature family",
    height=440,
)
fig.show()


In [ ]:
"""Velocity-trajectory comparison: monthly centroid drift, target vs control.

`monthly_centroid_velocities` measures how far each month's embedding centroid
moves from the prior month. Sustained higher velocity for the target indicates
faster stylistic drift than the control.
"""


def _velocities(slug: str) -> list[float]:
    p = paths.drift_json(slug)
    if not p.is_file():
        return []
    drift = json.loads(p.read_text(encoding="utf-8"))
    return list(drift.get("monthly_centroid_velocities") or [])


t_v = _velocities(target_slug)
c_v = _velocities(control_slug)
n = max(len(t_v), len(c_v))
x_idx = list(range(1, n + 1))

fig = go.Figure()
if t_v:
    fig.add_trace(
        go.Scatter(
            x=x_idx[: len(t_v)],
            y=t_v,
            mode="lines+markers",
            name=f"Target ({target_slug})",
            line=dict(color="#d62728", width=2),
        )
    )
if c_v:
    fig.add_trace(
        go.Scatter(
            x=x_idx[: len(c_v)],
            y=c_v,
            mode="lines+markers",
            name=f"Control ({control_slug})",
            line=dict(color="#1f77b4", width=2),
        )
    )
fig.update_layout(
    title="Monthly centroid-velocity trajectory (target vs control)",
    xaxis_title="Month index (1 = first observed month)",
    yaxis_title="Centroid velocity (1 - cosine similarity to prior month)",
    height=440,
)
fig.show()

if t_v and c_v:
    # Align by index for a like-for-like mean comparison.
    n_align = min(len(t_v), len(c_v))
    t_mean = sum(t_v[:n_align]) / n_align
    c_mean = sum(c_v[:n_align]) / n_align
    delta = t_mean - c_mean
    display(
        Markdown(
            f"Mean monthly velocity over the first **{n_align}** aligned months: "
            f"target = `{t_mean:.4f}`, control = `{c_mean:.4f}`, "
            f"Δ = `{delta:+.4f}` ({'target drifts faster' if delta > 0 else 'control drifts faster'})."
        )
    )


In [ ]:
"""Verdict: per-family two-sample test.

For each family, take every feature comparison from `feature_comparisons`
(target vs pooled controls Welch t-test, written by the analysis stage) and
declare the family "elevated in target" if at least one feature in the family
has p < 0.05 AND target_mean > control_mean.
"""

ALPHA = 0.05
elevated_families: list[str] = []
family_breakdown: dict[str, list[str]] = {f: [] for f in FAMILIES_ORDERED}

for feat, payload_block in feature_comparisons.items():
    family = FEATURE_FAMILIES.get(feat, "unknown")
    if family not in family_breakdown:
        continue
    stats_block = (payload_block or {}).get("all", {})
    p_val = stats_block.get("p_value")
    t_mean = stats_block.get("target_mean")
    c_mean = stats_block.get("control_mean")
    if p_val is None or t_mean is None or c_mean is None:
        continue
    if p_val < ALPHA and t_mean > c_mean:
        family_breakdown[family].append(feat)

elevated_families = [f for f in FAMILIES_ORDERED if family_breakdown[f]]
n_elevated = len(elevated_families)
n_total = len(FAMILIES_ORDERED)

bullets = "\n".join(
    f"- **{fam}** — elevated features: `{', '.join(family_breakdown[fam]) or '(none)'}`"
    for fam in FAMILIES_ORDERED
)

display(
    Markdown(
        f"""
## Verdict

**{n_elevated} of {n_total}** feature families show significantly higher activity
in the target (`{target_slug}`) than the control (`{control_slug}`)
at α = {ALPHA} (Welch t-test, target vs pooled controls).

**Editorial-vs-author signal** (from `comparison_report.json`):
`{editorial_vs_author_signal:.3f}` — closer to 1 means the target's change
points are *not* mirrored by the control, i.e. author-specific rather than
outlet-wide.

### Per-family breakdown

{bullets}
"""
    )
)


## Methodology note

Control comparison detects whether the target's signal is **anomalous relative
to a peer**. If the same change-point density, the same family elevations, and
the same centroid-velocity trajectory show up in a control author publishing
into the same outlet over the same period, the signal is editorial (CMS,
copy-desk, house style) — not the target's own writing change.

- **Pair selection:** the default pair (`colby-hall` target, `sarah-rumpf`
  control) is chosen because both authors have similar publishing volume but
  the target has the strongest convergence signal in the corpus. To pick a
  different pair, override the parameters cell:
  `quarto render notebooks/08_control_comparison.ipynb -P target_slug:foo -P control_slug:bar`.
- **Statistical test:** Welch's two-sample t-test per feature (target vs
  pooled controls). Family-level elevation requires at least one feature with
  `p < 0.05` AND `target_mean > control_mean`.
- **Editorial-vs-author signal:** for every target convergence window, count
  the fraction of controls with no overlapping window. Mean across all target
  windows. `0` = controls always agree (outlet-wide), `1` = controls never
  agree (author-specific).
- **Provenance:** when the persisted `comparison_report.json` does not cover
  the requested pair, this notebook regenerates it via
  `forensics.analysis.compare_target_to_controls` and writes the result back
  to disk.

**Summary finding:** Side-by-side control comparisons distinguish
author-specific drift from outlet-wide editorial shifts.
